In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import evaluate
import re

df = pd.read_csv('/content/domain_specific_chatbot_data.csv')
df.head()

,query,response,intent,domain
0,What are the side effects of the COVID-19 vacc...,Common side effects of the COVID-19 vaccine in...,side effects inquiry,healthcare
1,How can I schedule an appointment with my doctor?,You can schedule an appointment by calling our...,appointment booking,healthcare
2,What should I do if I miss a dose of my medica...,"If you miss a dose, take it as soon as you rem...",medication inquiry,healthcare
3,How can I check my account balance?,You can check your balance by logging into you...,balance inquiry,finance
4,What is the interest rate for a personal loan?,The current interest rate for personal loans i...,loan inquiry,finance


In [3]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_df.shape, val_df.shape

((2400, 4), (600, 4))

In [4]:
train_data = train_df.reset_index(drop=True)
val_data = val_df.reset_index(drop=True)

val_data

,query,response,intent,domain
0,How can I schedule an appointment with my doctor?,You can schedule an appointment by calling our...,appointment booking,healthcare
1,What are the side effects of the COVID-19 vacc...,Common side effects of the COVID-19 vaccine in...,side effects inquiry,healthcare
2,How do I update my contact details on my account?,"To update your contact details, log into your ...",contact update,finance
3,How can I schedule an appointment with my doctor?,You can schedule an appointment by calling our...,appointment booking,healthcare
4,"I lost my credit card, what should I do?",Please contact our customer service immediatel...,lost card reporting,finance
...,...,...,...,...
595,What is the interest rate for a personal loan?,The current interest rate for personal loans i...,loan inquiry,finance
596,How do I update my contact details on my account?,"To update your contact details, log into your ...",contact update,finance
597,How do I apply for a student loan?,You can apply for a student loan by visiting o...,student loan application,finance
598,What are the symptoms of flu?,"Flu symptoms include fever, cough, sore throat...",flu symptoms inquiry,healthcare


In [5]:
def clean_text(text):
  text = re.sub(r'\r\n', ' ', text)
  text = re.sub(r'\s+', ' ', text)
  text = re.sub(r'<.*?>', '', text)
  text = text.strip().lower()
  return text

In [6]:
train_data['query'] = train_data['query'].apply(clean_text)
train_data['response'] = train_data['response'].apply(clean_text)

val_data['query'] = val_data['query'].apply(clean_text)
val_data['response'] = val_data['response'].apply(clean_text)

train_data

,query,response,intent,domain
0,what should i do if i miss a dose of my medica...,"if you miss a dose, take it as soon as you rem...",medication inquiry,healthcare
1,what are the side effects of the covid-19 vacc...,common side effects of the covid-19 vaccine in...,side effects inquiry,healthcare
2,what are the symptoms of flu?,"flu symptoms include fever, cough, sore throat...",flu symptoms inquiry,healthcare
3,how do i update my contact details on my account?,"to update your contact details, log into your ...",contact update,finance
4,what are the side effects of the covid-19 vacc...,common side effects of the covid-19 vaccine in...,side effects inquiry,healthcare
...,...,...,...,...
2395,can i make changes to my loan repayment schedule?,changes to your loan repayment schedule can be...,loan repayment adjustment,finance
2396,"i lost my credit card, what should i do?",please contact our customer service immediatel...,lost card reporting,finance
2397,what are the side effects of the covid-19 vacc...,common side effects of the covid-19 vaccine in...,side effects inquiry,healthcare
2398,what is the interest rate for a personal loan?,the current interest rate for personal loans i...,loan inquiry,finance


In [7]:
tokenizer = T5Tokenizer.from_pretrained("t5-base")
def preprocess_func(example):
  inputs = tokenizer(example['query'], padding="max_length", truncation=True, max_length=250)
  targets = tokenizer(example['response'], padding="max_length", truncation=True, max_length=250)
  inputs["labels"] = targets["input_ids"]

  return inputs

train_dataset = train_data.apply(preprocess_func, axis=1)
val_dataset = val_data.apply(preprocess_func, axis=1)
train_data['response'][0], train_dataset[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

("if you miss a dose, take it as soon as you remember unless it's almost time for your next dose. if you’re unsure, contact your healthcare provider.",
 {'input_ids': [125, 225, 3, 23, 103, 3, 99, 3, 23, 3041, 3, 9, 6742, 13, 82, 7757, 58, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [8]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    save_steps=500,
    eval_steps=50,
    eval_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1800, training_loss=0.7790445312733452, metrics={'train_runtime': 622.0716, 'train_samples_per_second': 23.148, 'train_steps_per_second': 2.894, 'total_flos': 951622041600000.0, 'train_loss': 0.7790445312733452, 'epoch': 6.0})

In [9]:
model.save_pretrained('./chatbot_model')
tokenizer.save_pretrained('./chatbot_model')

model = T5ForConditionalGeneration.from_pretrained('./chatbot_model')
tokenizer = T5Tokenizer.from_pretrained('./chatbot_model')
device = model.device

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [10]:
def chatbot(query):
  query = clean_text(query)
  input_ids = tokenizer(query, return_tensors="pt", max_length=250, truncation=True)

  inputs = {
      key: value.to(device) for key, value in input_ids.items()
  }

  outputs = model.generate(
      inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=250,
      num_beams=5,
      early_stopping=True,
  )

  return tokenizer.decode(outputs[0], skip_special_tokens=True)

while True:
  user_input = input("You: ")
  if user_input.lower() in ["bye", "goodbye", "exit"]:
    print("Chatbot: Goodbye!")
    break
  response = chatbot(user_input)
  print("Chatbot:", response)

Chatbot: hi doctor!
Chatbot: you can visit our office or visit our office.
Chatbot: Goodbye!


In [11]:
import shutil
import os

shutil.make_archive('/content/trained_chatbot', 'zip', '/content/chatbot_model')

if os.path.exists('/content/trained_chatbot.zip'):
    print("Zip file successfully created!")
else:
    print("Zip file creation failed.")

Zip file successfully created!
